# 3-Statement Financial Model

This notebook demonstrates building a comprehensive 3-statement financial model including:
- **Income Statement** (Profit & Loss)
- **Balance Sheet** (Assets, Liabilities, Equity)
- **Cash Flow Statement** (Operating, Investing, Financing Activities)

In [ ]:
import pandas as pd
import json
from dataclasses import dataclass, asdict, field
from datetime import datetime
from typing import List, Optional
import csv
from pathlib import Path

In [ ]:
@dataclass
class Statement:
    """Class representing a financial statement line item."""
    id: str
    date: datetime
    amount: float
    description: str
    status: str = "active"
    category: Optional[str] = None
    
    def __post_init__(self):
        """Validate statement data after initialization."""
        if not isinstance(self.date, datetime):
            self.date = datetime.fromisoformat(str(self.date))
        if not isinstance(self.amount, (int, float)):
            raise TypeError(f"Amount must be numeric, got {type(self.amount)}")

print("Statement class defined successfully")

In [ ]:
# Create sample income statement items
income_stmt = [
    Statement(
        id="INC001",
        date=datetime(2025, 12, 31),
        amount=1000000,
        description="Total Revenue",
        status="active",
        category="Revenue"
    ),
    Statement(
        id="INC002",
        date=datetime(2025, 12, 31),
        amount=600000,
        description="Cost of Goods Sold",
        status="active",
        category="Expense"
    ),
    Statement(
        id="INC003",
        date=datetime(2025, 12, 31),
        amount=400000,
        description="Gross Profit",
        status="active",
        category="Intermediate"
    ),
]

# Create sample balance sheet items
balance_sheet = [
    Statement(
        id="BS001",
        date=datetime(2025, 12, 31),
        amount=500000,
        description="Cash and Equivalents",
        status="active",
        category="Asset"
    ),
    Statement(
        id="BS002",
        date=datetime(2025, 12, 31),
        amount=200000,
        description="Accounts Payable",
        status="active",
        category="Liability"
    ),
    Statement(
        id="BS003",
        date=datetime(2025, 12, 31),
        amount=800000,
        description="Shareholders' Equity",
        status="active",
        category="Equity"
    ),
]

print(f"Created {len(income_stmt)} income statement items")
print(f"Created {len(balance_sheet)} balance sheet items")

In [ ]:
def parse_statements_from_json(json_data: str) -> List[Statement]:
    """Parse statement objects from JSON data."""
    data = json.loads(json_data)
    statements = []
    for item in data:
        stmt = Statement(
            id=item['id'],
            date=datetime.fromisoformat(item['date']),
            amount=float(item['amount']),
            description=item['description'],
            status=item.get('status', 'active'),
            category=item.get('category')
        )
        statements.append(stmt)
    return statements

# Example JSON data
sample_json = '''[
    {"id": "CF001", "date": "2025-12-31", "amount": 150000, "description": "Operating Cash Flow", "status": "active", "category": "Operating"},
    {"id": "CF002", "date": "2025-12-31", "amount": -50000, "description": "Investing Activities", "status": "active", "category": "Investing"},
    {"id": "CF003", "date": "2025-12-31", "amount": 100000, "description": "Financing Activities", "status": "active", "category": "Financing"}
]'''

# Parse sample data
cash_flow_stmt = parse_statements_from_json(sample_json)
print(f"Parsed {len(cash_flow_stmt)} cash flow statement items from JSON")

# Display parsed data
for stmt in cash_flow_stmt[:2]:
    print(f"  - {stmt.description}: ${stmt.amount:,.0f}")

In [ ]:
def validate_statements(statements: List[Statement]) -> dict:
    """Validate a list of statements and return validation results."""
    validation_results = {
        'total_items': len(statements),
        'valid_items': 0,
        'invalid_items': 0,
        'errors': []
    }
    
    for i, stmt in enumerate(statements):
        try:
            # Check required fields
            assert stmt.id and isinstance(stmt.id, str), "Invalid ID"
            assert isinstance(stmt.date, datetime), "Invalid date"
            assert isinstance(stmt.amount, (int, float)), "Amount must be numeric"
            assert stmt.description and isinstance(stmt.description, str), "Invalid description"
            assert stmt.status in ['active', 'inactive', 'pending'], "Invalid status"
            
            validation_results['valid_items'] += 1
        except AssertionError as e:
            validation_results['invalid_items'] += 1
            validation_results['errors'].append({
                'index': i,
                'id': getattr(stmt, 'id', 'N/A'),
                'error': str(e)
            })
    
    return validation_results

# Validate all statements
all_statements = income_stmt + balance_sheet + cash_flow_stmt
validation = validate_statements(all_statements)

print("Validation Results:")
print(f"  Total Items: {validation['total_items']}")
print(f"  Valid Items: {validation['valid_items']}")
print(f"  Invalid Items: {validation['invalid_items']}")
print(f"  Status: {'✓ All valid' if validation['invalid_items'] == 0 else '✗ Some errors found'}")

In [ ]:
def export_to_json(statements: List[Statement], filename: str = "statements.json") -> str:
    """Export statements to JSON format."""
    data = []
    for stmt in statements:
        stmt_dict = asdict(stmt)
        stmt_dict['date'] = stmt_dict['date'].isoformat()
        data.append(stmt_dict)
    
    json_str = json.dumps(data, indent=2)
    print(f"Exported {len(statements)} statements to JSON")
    return json_str

def export_to_dataframe(statements: List[Statement]) -> pd.DataFrame:
    """Export statements to pandas DataFrame."""
    data = []
    for stmt in statements:
        stmt_dict = asdict(stmt)
        data.append(stmt_dict)
    
    df = pd.DataFrame(data)
    print(f"Exported {len(statements)} statements to DataFrame")
    return df

def export_to_csv(statements: List[Statement], filename: str = "statements.csv") -> None:
    """Export statements to CSV format."""
    if len(statements) == 0:
        print("No statements to export")
        return
    
    with open(filename, 'w', newline='') as csvfile:
        fieldnames = ['id', 'date', 'amount', 'description', 'status', 'category']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for stmt in statements:
            row = {
                'id': stmt.id,
                'date': stmt.date.isoformat(),
                'amount': stmt.amount,
                'description': stmt.description,
                'status': stmt.status,
                'category': stmt.category
            }
            writer.writerow(row)
    
    print(f"Exported {len(statements)} statements to CSV: {filename}")

# Export sample data
json_output = export_to_json(all_statements)
df_output = export_to_dataframe(all_statements)

print("\n--- Sample JSON Export (first 2 items) ---")
print(json_output[:200] + "...")

print("\n--- DataFrame Summary ---")
print(df_output[['id', 'description', 'amount']].head())

## Section 6: Export Statement Model

Export the statement model to various formats (JSON, CSV, or serialized objects).

## Section 5: Validate Statement Structure

Implement validation logic to ensure statement data meets required constraints.

## Section 4: Parse Statement Data

Load and parse statement data from various sources (currently using sample data).

## Section 3: Create Statement Instances

Instantiate multiple Statement objects with sample data.

## Section 2: Define Statement Class

Create a Statement class with attributes for financial statement representation.

## Section 1: Import Required Libraries

Import necessary libraries for data handling and model definition.